in this notebook we will evaluate the m-e5 model that we are using in parsaqa website

for its semantic search capabilities

using the dedicated libraries that we have developed for the tasks

for searching we will use `ai` library from ai_services repo

fro evaluation we will use `evaluator` library, whihc is a custom library we created specific for reranking search engines


first we install the libraries

In [1]:
# !pip install git+https://github.com/HadithAi/ai_services.git

In [2]:
# !https://github.com/HadithAi/evaluator

In [3]:
from elasticsearch import AsyncElasticsearch as Elasticsearch
import tritonclient.http as httpclient
from dotenv import load_dotenv

# our custom libraries
from ai import CosineSemanticSearch

import os

load_dotenv()

True

we need triton client for semantic search

In [4]:
triton_client = httpclient.InferenceServerClient(url="172.30.0.113:8000")

In [5]:
# config for the semantic searcher
config = {
    "es_client": Elasticsearch(
            hosts=[{
                'host': os.getenv('ES_URL'),
                'port': 9200,
                'scheme': 'https'
            }],
            basic_auth=(os.getenv('ES_USER'), os.getenv('ES_PASS')),
            verify_certs=False
        ),
    "index_name": "parsaqa_questions",
    "triton_client": triton_client,
    "triton_instruction": """
    retrieve the closest question to this question
    """,
    "triton_model_name": "e5"
    
}

/home/tohidi/miniconda3/envs/general/lib/python3.10/site-packages/elasticsearch/_async/client/__init__.py:403: SecurityWarning: Connecting to 'https://172.30.0.114:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


we define the Search Engine here

In [6]:
search_engine = CosineSemanticSearch(config=config)

now we need a list of good questions

that we would want to rerank and evaluate our system based on them

In [7]:
evaluate_questions = [
    """
    نماز صبح چند رکعت است؟
    """,
    """
    گراز
    """,
    """
    الکل در شکلات
    """,
    """
	خوردن کانگورو چه حکمی داره؟
    """,
	"""
	در قرآن چه مطالبی درباره کره زمین آمده است؟
 	"""
 
]

In [18]:
questions_results = []

for idx, question in enumerate(evaluate_questions):
	search_res = await search_engine.search(query=question, size=10, language="fa", source=True)
	ids = search_res['result']
    
	items = []
 
	print(search_res["content"][idx]["_source"])
    
	for id, item in zip(ids, search_res["content"][idx]["_source"]):
		print(item['question'])
     
     
		items.append(
      {
		"id": id,
		"body": item['question']['text']['fa']
	  }
      
      ) 
    	
	questions_results.append({"query": question, "items": items})

{'metadata': {'flat_category': [], 'categories': [], 'source': {'persian_name': 'پرسمان دانشگاهیان', 'english_name': 'porseman', 'url': 'https://www.porseman.com'}, 'public_figure': None, 'tags': []}, 'view': 0, 'question': {'bge_serach_vector': [-0.007206345442682505, 0.02995915897190571, -0.03795071691274643, 0.003758520120754838, -0.030042776837944984, 0.0016948783304542303, -0.02426799386739731, -0.006498463451862335, 0.000722715980373323, -0.020665563642978668, -0.045551177114248276, 0.024849148467183113, -0.054640281945466995, -0.00023693391995038837, 0.037900410592556, -0.0033690754789859056, 0.031040942296385765, -0.043101366609334946, 0.024303773418068886, -0.002134075155481696, 0.040115728974342346, 0.002701910911127925, -0.006311371922492981, 0.04770669341087341, 0.006533592939376831, 0.022659746930003166, -0.052732378244400024, -0.024654947221279144, 0.06128740310668945, 0.02189982496201992, -0.03018234111368656, -0.015926534309983253, -0.001991389552131295, -0.041464742273

TypeError: string indices must be integers